# Генерация заголовков научных статей: слабый baseline

Источник: https://github.com/bentrevett/pytorch-seq2seq

In [5]:
# Если Вы запускаете ноутбук на colab,
# выполните следующие строчки, чтобы подгрузить библиотеку dlnlputils:

!git clone https://github.com/Samsung-IT-Academy/stepik-dl-nlp.git
import sys; sys.path.append('/content/stepik-dl-nlp')

fatal: destination path 'stepik-dl-nlp' already exists and is not an empty directory.


In [6]:
!pip install torch==1.8.1+cu111 torchtext==0.9.1 -f https://download.pytorch.org/whl/torch_stable.html

Looking in links: https://download.pytorch.org/whl/torch_stable.html
ERROR: Could not find a version that satisfies the requirement torch==1.8.1+cu111 (from versions: 2.2.0, 2.2.0+cpu, 2.2.0+cpu.cxx11.abi, 2.2.0+cu118, 2.2.0+cu121, 2.2.0+rocm5.6, 2.2.0+rocm5.7, 2.2.1, 2.2.1+cpu, 2.2.1+cpu.cxx11.abi, 2.2.1+cu118, 2.2.1+cu121, 2.2.1+rocm5.6, 2.2.1+rocm5.7, 2.2.2, 2.2.2+cpu, 2.2.2+cpu.cxx11.abi, 2.2.2+cu118, 2.2.2+cu121, 2.2.2+rocm5.6, 2.2.2+rocm5.7, 2.3.0, 2.3.0+cpu, 2.3.0+cpu.cxx11.abi, 2.3.0+cu118, 2.3.0+cu121, 2.3.0+rocm5.7, 2.3.0+rocm6.0, 2.3.1, 2.3.1+cpu, 2.3.1+cpu.cxx11.abi, 2.3.1+cu118, 2.3.1+cu121, 2.3.1+rocm5.7, 2.3.1+rocm6.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==1.8.1+cu111


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchtext.data import Field, BucketIterator

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import spacy

import random
import math
import time

In [ ]:
SEED = 1234

random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
# возможно, Вам потребуется предварительно загрузить модели SpaCy для английского языка
# !python -m spacy download en

spacy_en = spacy.load('en')

In [ ]:
def tokenize(text):
    """
    Tokenizes English text from a string into a list of strings (tokens)
    """
    return [tok.text for tok in spacy_en.tokenizer(text) if not tok.text.isspace()]

In [ ]:
from torchtext import data, vocab

tokenizer = data.get_tokenizer('spacy')
TEXT = Field(tokenize=tokenize,
            init_token = '<sos>',
            eos_token = '<eos>',
            include_lengths = True,
            lower = True)



In [ ]:
%%time
trn_data_fields = [("src", TEXT),
                   ("trg", TEXT)]

dataset = data.TabularDataset(
    path='datasets/train.csv',
    format='csv',
    skip_header=True,
    fields=trn_data_fields
)

train_data, valid_data, test_data = dataset.split(split_ratio=[0.98, 0.01, 0.01])

In [ ]:
TEXT.build_vocab(train_data, min_freq = 7)
print(f"Unique tokens in vocabulary: {len(TEXT.vocab)}")

In [ ]:
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_iterator, valid_iterator, test_iterator = BucketIterator.splits(
    (train_data, valid_data, test_data),
     batch_size = BATCH_SIZE,
     sort_within_batch = True,
     sort_key = lambda x : len(x.src),
     device = device)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional = True)

        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_len):

        #src = [src sent len, batch size]
        #src_len = [src sent len]

        embedded = self.dropout(self.embedding(src))

        #embedded = [src sent len, batch size, emb dim]

        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, src_len)

        packed_outputs, hidden = self.rnn(packed_embedded)

        #packed_outputs is a packed sequence containing all hidden states
        #hidden is now from the final non-padded element in the batch

        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs)

        #outputs is now a non-packed sequence, all hidden states obtained
        #  when the input is a pad token are all zeros

        #outputs = [sent len, batch size, hid dim * num directions]
        #hidden = [n layers * num directions, batch size, hid dim]

        #hidden is stacked [forward_1, backward_1, forward_2, backward_2, ...]
        #outputs are always from the last layer

        #hidden [-2, :, : ] is the last of the forwards RNN
        #hidden [-1, :, : ] is the last of the backwards RNN

        #initial decoder hidden is final hidden state of the forwards and backwards
        #  encoder RNNs fed through a linear layer
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim = 1)))

        #outputs = [sent len, batch size, enc hid dim * 2]
        #hidden = [batch size, dec hid dim]

        return outputs, hidden

In [ ]:
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()

        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Parameter(torch.rand(dec_hid_dim))

    def forward(self, hidden, encoder_outputs, mask):

        #hidden = [batch size, dec hid dim]
        #encoder_outputs = [src sent len, batch size, enc hid dim * 2]
        #mask = [batch size, src sent len]

        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        #repeat encoder hidden state src_len times
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        #hidden = [batch size, src sent len, dec hid dim]
        #encoder_outputs = [batch size, src sent len, enc hid dim * 2]

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim = 2)))

        #energy = [batch size, src sent len, dec hid dim]

        energy = energy.permute(0, 2, 1)

        #energy = [batch size, dec hid dim, src sent len]

        #v = [dec hid dim]

        v = self.v.repeat(batch_size, 1).unsqueeze(1)

        #v = [batch size, 1, dec hid dim]

        attention = torch.bmm(v, energy).squeeze(1)

        #attention = [batch size, src sent len]

        attention = attention.masked_fill(mask == 0, -1e10)

        return F.softmax(attention, dim = 1)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim)

        self.out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs, mask):

        #input = [batch size]
        #hidden = [batch size, dec hid dim]
        #encoder_outputs = [src sent len, batch size, enc hid dim * 2]
        #mask = [batch size, src sent len]

        input = input.unsqueeze(0)

        #input = [1, batch size]

        embedded = self.dropout(self.embedding(input))

        #embedded = [1, batch size, emb dim]

        a = self.attention(hidden, encoder_outputs, mask)

        #a = [batch size, src sent len]

        a = a.unsqueeze(1)

        #a = [batch size, 1, src sent len]

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        #encoder_outputs = [batch size, src sent len, enc hid dim * 2]

        weighted = torch.bmm(a, encoder_outputs)

        #weighted = [batch size, 1, enc hid dim * 2]

        weighted = weighted.permute(1, 0, 2)

        #weighted = [1, batch size, enc hid dim * 2]

        rnn_input = torch.cat((embedded, weighted), dim = 2)

        #rnn_input = [1, batch size, (enc hid dim * 2) + emb dim]

        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))

        #output = [sent len, batch size, dec hid dim * n directions]
        #hidden = [n layers * n directions, batch size, dec hid dim]

        #sent len, n layers and n directions will always be 1 in this decoder, therefore:
        #output = [1, batch size, dec hid dim]
        #hidden = [1, batch size, dec hid dim]
        #this also means that output == hidden
        assert (output == hidden).all()

        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)

        output = self.out(torch.cat((output, weighted, embedded), dim = 1))

        #output = [bsz, output dim]

        return output, hidden.squeeze(0), a.squeeze(1)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_idx, sos_idx, eos_idx, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.sos_idx = sos_idx
        self.eos_idx = eos_idx
        self.device = device

    def create_mask(self, src):
        mask = (src != self.pad_idx).permute(1, 0)
        return mask

    def forward(self, src, src_len, trg, teacher_forcing_ratio = 0.5):

        #src = [src sent len, batch size]
        #src_len = [batch size]
        #trg = [trg sent len, batch size]
        #teacher_forcing_ratio is probability to use teacher forcing
        #e.g. if teacher_forcing_ratio is 0.75 we use teacher forcing 75% of the time

        if trg is None:
            assert teacher_forcing_ratio == 0, "Must be zero during inference"
            inference = True
            trg = torch.zeros((100, src.shape[1])).long().fill_(self.sos_idx).to(src.device)
        else:
            inference = False

        batch_size = src.shape[1]
        max_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        #tensor to store decoder outputs
        outputs = torch.zeros(max_len, batch_size, trg_vocab_size).to(self.device)

        #tensor to store attention
        attentions = torch.zeros(max_len, batch_size, src.shape[0]).to(self.device)

        #encoder_outputs is all hidden states of the input sequence, back and forwards
        #hidden is the final forward and backward hidden states, passed through a linear layer
        encoder_outputs, hidden = self.encoder(src, src_len)

        #first input to the decoder is the <sos> tokens
        input = trg[0,:]

        mask = self.create_mask(src)

        #mask = [batch size, src sent len]

        for t in range(1, max_len):

            #insert input token embedding, previous hidden state, all encoder hidden states
            # and mask
            #receive output tensor (predictions), new hidden state and attention tensor
            output, hidden, attention = self.decoder(input, hidden, encoder_outputs, mask)

            #place predictions in a tensor holding predictions for each token
            outputs[t] = output

            #place attentions in a tensor holding attention value for each input token
            attentions[t] = attention

            #decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            #get the highest predicted token from our predictions
            top1 = output.argmax(1)

            #if teacher forcing, use actual next token as next input
            #if not, use predicted token
            input = trg[t] if teacher_force else top1

            #if doing inference and next token/prediction is an eos token then stop
            if inference and input.item() == self.eos_idx:
                return outputs[:t], attentions[:t]

        return outputs, attentions

In [ ]:
INPUT_DIM = len(TEXT.vocab)
OUTPUT_DIM = len(TEXT.vocab)
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 64
DEC_HID_DIM = 64
ENC_DROPOUT = 0.8
DEC_DROPOUT = 0.8
PAD_IDX = TEXT.vocab.stoi['<pad>']
SOS_IDX = TEXT.vocab.stoi['<sos>']
EOS_IDX = TEXT.vocab.stoi['<eos>']

attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)

model = Seq2Seq(enc, dec, PAD_IDX, SOS_IDX, EOS_IDX, device).to(device)

In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)

model.apply(init_weights)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

In [ ]:
optimizer = optim.Adam(model.parameters())

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index = PAD_IDX)

### Обучение модели

In [ ]:
import matplotlib
matplotlib.rcParams.update({'figure.figsize': (16, 12), 'font.size': 14})
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output


def train(model, iterator, optimizer, criterion, clip, train_history=None, valid_history=None):

    model.train()

    epoch_loss = 0
    history = []
    for i, batch in enumerate(iterator):

        src, src_len = batch.src
        trg, trg_len = batch.trg

        optimizer.zero_grad()

        output, attetion = model(src, src_len, trg)

        #trg = [trg sent len, batch size]
        #output = [trg sent len, batch size, output dim]

        output = output[1:].view(-1, output.shape[-1])
        trg = trg[1:].view(-1)

        #trg = [(trg sent len - 1) * batch size]
        #output = [(trg sent len - 1) * batch size, output dim]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

        history.append(loss.cpu().data.numpy())
        if (i+1)%10==0:
            fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 8))

            clear_output(True)
            ax[0].plot(history, label='train loss')
            ax[0].set_xlabel('Batch')
            ax[0].set_title('Train loss')
            if train_history is not None:
                ax[1].plot(train_history, label='general train history')
                ax[1].set_xlabel('Epoch')
            if valid_history is not None:
                ax[1].plot(valid_history, label='general valid history')
            plt.legend()

            plt.show()

    return epoch_loss / len(iterator)

In [ ]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for i, batch in enumerate(iterator):

            src, src_len = batch.src
            trg, trg_len = batch.trg

            output, attention = model(src, src_len, trg, 0) #turn off teacher forcing

            #trg = [trg sent len, batch size]
            #output = [trg sent len, batch size, output dim]

            output = output[1:].view(-1, output.shape[-1])
            trg = trg[1:].view(-1)

            #trg = [(trg sent len - 1) * batch size]
            #output = [(trg sent len - 1) * batch size, output dim]

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
MODEL_NAME = 'models/lstm_baseline.pt'
N_EPOCHS = 5
CLIP = 1

train_history = []
valid_history = []

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_iterator, optimizer, criterion, CLIP, train_history, valid_history)
    valid_loss = evaluate(model, valid_iterator, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), MODEL_NAME)


    train_history.append(train_loss)
    valid_history.append(valid_loss)

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Finally, we load the parameters from our best validation loss and get our results on the test set.

In [ ]:
# for cpu usage
model.load_state_dict(torch.load(MODEL_NAME, map_location=torch.device('cpu')))

# for gpu usage
# model.load_state_dict(torch.load(MODEL_NAME), map_location=torch.device('cpu'))


test_loss = evaluate(model, test_iterator, criterion)

print(f'| Test Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f} |')

### Генерация заголовков

In [ ]:
def translate_sentence(model, tokenized_sentence):
    model.eval()
    tokenized_sentence = ['<sos>'] + [t.lower() for t in tokenized_sentence] + ['<eos>']
    numericalized = [TEXT.vocab.stoi[t] for t in tokenized_sentence]
    sentence_length = torch.LongTensor([len(numericalized)]).to(device)
    tensor = torch.LongTensor(numericalized).unsqueeze(1).to(device)
    translation_tensor_logits, attention = model(tensor, sentence_length, None, 0)
    translation_tensor = torch.argmax(translation_tensor_logits.squeeze(1), 1)
    translation = [TEXT.vocab.itos[t] for t in translation_tensor]
    translation, attention = translation[1:], attention[1:]
    return translation, attention

In [ ]:
def display_attention(sentence, translation, attention):

    fig = plt.figure(figsize=(30,50))
    ax = fig.add_subplot(111)

    attention = attention.squeeze(1).cpu().detach().numpy().T

    cax = ax.matshow(attention, cmap='bone')

    ax.tick_params(labelsize=12)
    ax.set_yticklabels(['']+['<sos>']+[t.lower() for t in sentence]+['<eos>'])
    ax.set_xticklabels(['']+translation, rotation=80)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()
    plt.close()

In [ ]:
example_idx = 100

src = vars(train_data.examples[example_idx])['src']
trg = vars(train_data.examples[example_idx])['trg']

print(f'src = {src}')
print(f'trg = {trg}')

In [ ]:
translation, attention = translate_sentence(model, src)

print(f'predicted trg = {translation}')

In [ ]:
display_attention(src, translation, attention)

In [ ]:
for example_idx in range(100):
    src = vars(test_data.examples[example_idx])['src']
    trg = vars(test_data.examples[example_idx])['trg']
    translation, attention = translate_sentence(model, src)

    print('Оригинальный заголовок: ', ' '.join(trg))
    print('Предсказанный заголовок: ', ' '.join(translation))
    print('-----------------------------------')

In [ ]:
example_idx = 0

src = vars(valid_data.examples[example_idx])['src']
trg = vars(valid_data.examples[example_idx])['trg']

print(f'src = {src}')
print(f'trg = {trg}')

In [ ]:
translation, attention = translate_sentence(model, src)

print(f'predicted trg = {translation}')

display_attention(src, translation, attention)

In [ ]:
example_idx = 510

src = vars(test_data.examples[example_idx])['src']
trg = vars(test_data.examples[example_idx])['trg']

print(f'src = {src}')
print(f'trg = {trg}')

In [ ]:
translation, attention = translate_sentence(model, src)

print(f'predicted trg = {translation}')

display_attention(src, translation, attention)

### Считаем BLEU на train.csv

In [ ]:
import nltk

n_gram_weights = [0.3334, 0.3333, 0.3333]

In [ ]:
test_len = len(test_data)

In [ ]:
original_texts = []
generated_texts = []
macro_bleu = 0

for example_idx in range(test_len):
    src = vars(test_data.examples[example_idx])['src']
    trg = vars(test_data.examples[example_idx])['trg']
    translation, _ = translate_sentence(model, src)

    original_texts.append(trg)
    generated_texts.append(translation)

    bleu_score = nltk.translate.bleu_score.sentence_bleu(
        [trg],
        translation,
        weights = n_gram_weights
    )
    macro_bleu += bleu_score

macro_bleu /= test_len

In [ ]:
# averaging sentence-level BLEU (i.e. macro-average precision)
print('Macro-average BLEU (LSTM): {0:.5f}'.format(macro_bleu))

### Делаем submission в Kaggle

In [ ]:
import pandas as pd

submission_data = pd.read_csv('datasets/test.csv')
abstracts = submission_data['abstract'].values

Генерация заголовков для тестовых данных:

In [ ]:
titles = []
for abstract in abstracts:
    title, _ = translate_sentence(model, abstract.split())
    titles.append(' '.join(title).replace('<unk>', ''))

Записываем полученные заголовки в файл формата `<abstract>,<title>`:

In [ ]:
submission_df = pd.DataFrame({'abstract': abstracts, 'title': titles})
submission_df.to_csv('datasets/predicted_titles.csv', index=False)

С помощью скрипта `generate_csv` приводим файл `submission_prediction.csv` в формат, необходимый для посылки в соревнование на Kaggle:

In [ ]:
from create_submission import generate_csv

generate_csv('datasets/predicted_titles.csv', 'datasets/kaggle_pred.csv', 'datasets/vocs.pkl')

In [ ]:
!wc -l datasets/kaggle_pred.csv

In [ ]:
!head datasets/kaggle_pred.csv

In [1]:
# -*- coding: utf-8 -*-
"""
LSTM Baseline for Scientific Title Generation - Inference Only Version
Original source: https://github.com/bentrevett/pytorch-seq2seq

This version:
- Loads pre-trained weights
- Evaluates model performance
- Calculates F1-score (target: 50%)
- Works with manually uploaded CSV files in Google Colab
"""

import os
import sys
import random
import math
import time
from typing import List, Tuple, Optional
from sklearn.metrics import f1_score
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import matplotlib
matplotlib.rcParams.update({'figure.figsize': (16, 12), 'font.size': 14})
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import spacy
import nltk
from nltk.translate.bleu_score import sentence_bleu
import pandas as pd
from google.colab import files

# Set random seeds for reproducibility
SEED = 1234
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================================
# Data Preparation
# ============================================================================

class Vocabulary:
    """Custom vocabulary class"""
    def __init__(self, freq_threshold=7):
        self.itos = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.stoi = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocab(self, sentence_list: List[List[str]]):
        """Build vocabulary from list of tokenized sentences"""
        frequencies = {}
        idx = 4

        for sentence in sentence_list:
            for word in sentence:
                if word not in frequencies:
                    frequencies[word] = 1
                else:
                    frequencies[word] += 1

        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, text: List[str]) -> List[int]:
        """Convert tokenized text to numerical indices"""
        return [self.stoi.get(token, self.stoi['<unk>']) for token in text]

class TitleGenerationDataset(Dataset):
    """Custom dataset for title generation"""
    def __init__(self, src_data: List[List[str]], trg_data: List[List[str]], src_vocab: Vocabulary, trg_vocab: Vocabulary):
        self.src_data = src_data
        self.trg_data = trg_data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        src = self.src_data[idx]
        trg = self.trg_data[idx]

        src_numericalized = [self.src_vocab.stoi['<sos>']] + \
                           self.src_vocab.numericalize(src) + \
                           [self.src_vocab.stoi['<eos>']]
        trg_numericalized = [self.trg_vocab.stoi['<sos>']] + \
                           self.trg_vocab.numericalize(trg) + \
                           [self.trg_vocab.stoi['<eos>']]

        return torch.tensor(src_numericalized, dtype=torch.long), \
               torch.tensor(trg_numericalized, dtype=torch.long)

def collate_fn(batch: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Collate function for DataLoader"""
    src_batch, trg_batch = zip(*batch)

    src_padded = pad_sequence(src_batch, padding_value=0, batch_first=False)
    trg_padded = pad_sequence(trg_batch, padding_value=0, batch_first=False)

    src_lengths = torch.tensor([len(x) for x in src_batch], dtype=torch.long)
    trg_lengths = torch.tensor([len(x) for x in trg_batch], dtype=torch.long)

    return src_padded, src_lengths, trg_padded, trg_lengths

# ============================================================================
# Tokenization
# ============================================================================

def load_spacy_model():
    """Load spaCy English model"""
    try:
        spacy_en = spacy.load('en_core_web_sm')
    except OSError:
        print("Downloading spaCy English model...")
        os.system('python -m spacy download en_core_web_sm')
        spacy_en = spacy.load('en_core_web_sm')
    return spacy_en

def tokenize(text: str, tokenizer) -> List[str]:
    """Tokenize English text"""
    return [tok.text for tok in tokenizer(text) if not tok.text.isspace()]

# ============================================================================
# Improved Model Architecture (for better F1-score)
# ============================================================================

class Encoder(nn.Module):
    def __init__(self, input_dim: int, emb_dim: int, enc_hid_dim: int, dec_hid_dim: int, dropout: float, n_layers: int = 2):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, num_layers=n_layers, bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(enc_hid_dim * 2)

    def forward(self, src: torch.Tensor, src_len: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(src))

        packed_embedded = pack_padded_sequence(embedded, src_len, enforce_sorted=False)
        packed_outputs, hidden = self.rnn(packed_embedded)

        outputs, _ = pad_packed_sequence(packed_outputs)
        outputs = self.layer_norm(outputs)

        # Combine bidirectional hidden states
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))

        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, enc_hid_dim: int, dec_hid_dim: int):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Parameter(torch.rand(dec_hid_dim))

    def forward(self, hidden: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        energy = energy.permute(0, 2, 1)

        v = self.v.repeat(batch_size, 1).unsqueeze(1)
        attention = torch.bmm(v, energy).squeeze(1)

        attention = attention.masked_fill(mask == 0, -1e10)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim: int, emb_dim: int, enc_hid_dim: int, dec_hid_dim: int, dropout: float, attention: nn.Module, n_layers: int = 2):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim, num_layers=n_layers, dropout=dropout if n_layers > 1 else 0)
        self.out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(dec_hid_dim)

    def forward(self, input: torch.Tensor, hidden: torch.Tensor, encoder_outputs: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))

        a = self.attention(hidden, encoder_outputs, mask)
        a = a.unsqueeze(1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs)
        weighted = weighted.permute(1, 0, 2)

        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))

        # Apply layer normalization
        output = self.layer_norm(output.squeeze(0))
        hidden = self.layer_norm(hidden.squeeze(0))

        embedded = embedded.squeeze(0)
        weighted = weighted.squeeze(0)

        output = self.out(torch.cat((output, weighted, embedded), dim=1))

        return output, hidden, a.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, encoder: nn.Module, decoder: nn.Module, pad_idx: int, sos_idx: int, eos_idx: int, device: torch.device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.sos_idx = sos_idx
        self.eos_idx = eos_idx
        self.device = device

    def create_mask(self, src: torch.Tensor) -> torch.Tensor:
        mask = (src != self.pad_idx).permute(1, 0)
        return mask

    def forward(self, src: torch.Tensor, src_len: torch.Tensor, trg: Optional[torch.Tensor] = None, teacher_forcing_ratio: float = 0.0) -> Tuple[torch.Tensor, torch.Tensor]:
        if trg is None:
            inference = True
            trg = torch.zeros((100, src.shape[1]), dtype=torch.long).fill_(self.sos_idx).to(src.device)
        else:
            inference = False

        batch_size = src.shape[1]
        max_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(max_len, batch_size, trg_vocab_size).to(self.device)
        attentions = torch.zeros(max_len, batch_size, src.shape[0]).to(self.device)

        encoder_outputs, hidden = self.encoder(src, src_len)
        input = trg[0,:]
        mask = self.create_mask(src)

        for t in range(1, max_len):
            output, hidden, attention = self.decoder(input, hidden, encoder_outputs, mask)
            outputs[t] = output
            attentions[t] = attention

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

            if inference and input.item() == self.eos_idx:
                return outputs[:t], attentions[:t]

        return outputs, attentions

# ============================================================================
# Evaluation Metrics
# ============================================================================

def calculate_f1_score(references: List[List[str]], predictions: List[List[str]]) -> float:
    """Calculate macro F1-score for token-level comparison"""
    all_ref_tokens = []
    all_pred_tokens = []

    for ref, pred in zip(references, predictions):
        # Convert to sets of unique tokens for each sentence
        ref_set = set(ref)
        pred_set = set(pred)

        # Calculate precision and recall for this pair
        common = ref_set.intersection(pred_set)
        precision = len(common) / len(pred_set) if len(pred_set) > 0 else 0
        recall = len(common) / len(ref_set) if len(ref_set) > 0 else 0

        # F1 for this pair
        if (precision + recall) > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0

        all_ref_tokens.extend(ref)
        all_pred_tokens.extend(pred)

    # Also calculate overall F1 using sklearn
    try:
        # Convert to binary vectors (presence/absence of each token)
        all_tokens = list(set(all_ref_tokens + all_pred_tokens))
        token_to_idx = {token: idx for idx, token in enumerate(all_tokens)}

        # Create binary matrices
        ref_matrix = np.zeros((len(references), len(all_tokens)))
        pred_matrix = np.zeros((len(predictions), len(all_tokens)))

        for i, ref in enumerate(references):
            for token in ref:
                if token in token_to_idx:
                    ref_matrix[i, token_to_idx[token]] = 1

        for i, pred in enumerate(predictions):
            for token in pred:
                if token in token_to_idx:
                    pred_matrix[i, token_to_idx[token]] = 1

        # Calculate F1-score (macro average)
        f1_macro = f1_score(ref_matrix, pred_matrix, average='macro', zero_division=0)
        return f1_macro * 100  # Return as percentage
    except:
        return 0.0

# ============================================================================
# Translation and Attention Visualization
# ============================================================================

def translate_sentence(model: nn.Module, sentence: List[str], src_vocab: Vocabulary, trg_vocab: Vocabulary, max_len: int = 100) -> Tuple[List[str], torch.Tensor]:
    """Translate a sentence using the model"""
    model.eval()

    tokenized = ['<sos>'] + [t.lower() for t in sentence] + ['<eos>']
    numericalized = [src_vocab.stoi.get(t, src_vocab.stoi['<unk>']) for t in tokenized]

    sentence_length = torch.LongTensor([len(numericalized)]).to(device)
    tensor = torch.LongTensor(numericalized).unsqueeze(1).to(device)

    with torch.no_grad():
        translation_tensor_logits, attention = model(tensor, sentence_length, None, 0)

    translation_tensor = torch.argmax(translation_tensor_logits.squeeze(1), 1)
    translation = [trg_vocab.itos.get(t.item(), '<unk>') for t in translation_tensor]
    translation, attention = translation[1:], attention[1:]

    return translation, attention

def display_attention(sentence: List[str], translation: List[str], attention: torch.Tensor):
    """Display attention weights"""
    fig = plt.figure(figsize=(30, 50))
    ax = fig.add_subplot(111)

    attention = attention.squeeze(1).cpu().detach().numpy().T
    cax = ax.matshow(attention, cmap='bone')

    ax.tick_params(labelsize=12)
    ax.set_yticklabels([''] + ['<sos>'] + [t.lower() for t in sentence] + ['<eos>'])
    ax.set_xticklabels([''] + translation, rotation=80)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()
    plt.close()

# ============================================================================
# Main Execution
# ============================================================================

def load_and_preprocess_data(train_path: str, test_path: str) -> Tuple[TitleGenerationDataset, TitleGenerationDataset, Vocabulary, Vocabulary]:
    """Load and preprocess data"""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    spacy_en = load_spacy_model()

    # Prepare source and target data
    train_src_data = [tokenize(text, spacy_en) for text in train_df['abstract'].tolist()]
    train_trg_data = [tokenize(text, spacy_en) for text in train_df['title'].tolist()]

    test_src_data = [tokenize(text, spacy_en) for text in test_df['abstract'].tolist()]
    test_trg_data = [tokenize(text, spacy_en) for text in test_df['title'].tolist()]

    # Build vocabularies
    src_vocab = Vocabulary(freq_threshold=7)
    trg_vocab = Vocabulary(freq_threshold=7)

    # Build vocab from training data
    src_vocab.build_vocab(train_src_data)
    trg_vocab.build_vocab(train_trg_data)

    # Create datasets
    train_dataset = TitleGenerationDataset(train_src_data, train_trg_data, src_vocab, trg_vocab)
    test_dataset = TitleGenerationDataset(test_src_data, test_trg_data, src_vocab, trg_vocab)

    return train_dataset, test_dataset, src_vocab, trg_vocab

def upload_files():
    """Upload files in Google Colab"""
    print("\n" + "="*60)
    print("PLEASE UPLOAD THE FOLLOWING FILES:")
    print("1. train.csv (with 'abstract' and 'title' columns)")
    print("2. test.csv (with 'abstract' column)")
    print("3. lstm_baseline.pt (pre-trained model weights)")
    print("="*60 + "\n")

    # Create datasets directory
    os.makedirs('datasets', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    # Upload train.csv
    if not os.path.exists('datasets/train.csv'):
        print("Upload train.csv...")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename == 'train.csv':
                os.rename(filename, 'datasets/train.csv')
                print(f"✓ train.csv saved to datasets/train.csv")

    # Upload test.csv
    if not os.path.exists('datasets/test.csv'):
        print("\nUpload test.csv...")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename == 'test.csv':
                os.rename(filename, 'datasets/test.csv')
                print(f"✓ test.csv saved to datasets/test.csv")

    # Upload model weights
    if not os.path.exists('models/lstm_baseline.pt'):
        print("\nUpload pre-trained model weights (lstm_baseline.pt)...")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename == 'lstm_baseline.pt':
                os.rename(filename, 'models/lstm_baseline.pt')
                print(f"✓ lstm_baseline.pt saved to models/lstm_baseline.pt")

def main():
    """Main inference function"""
    print("="*60)
    print("LSTM BASELINE FOR TITLE GENERATION - INFERENCE MODE")
    print("="*60)

    # Install required packages
    try:
        import spacy
        import nltk
        import sklearn
    except ImportError:
        print("Installing required packages...")
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'spacy', 'nltk', 'scikit-learn', 'pandas', 'matplotlib'], check=True)
        subprocess.run(['python', '-m', 'spacy', 'download', '-q', 'en_core_web_sm'], check=True)
        subprocess.run(['python', '-m', 'nltk.downloader', '-q', 'punkt'], check=True)

    # Upload files in Colab
    if 'google.colab' in sys.modules:
        upload_files()

    # Check if files exist
    if not os.path.exists('datasets/train.csv'):
        print("ERROR: datasets/train.csv not found!")
        print("Please upload the file or check the path.")
        return

    if not os.path.exists('datasets/test.csv'):
        print("ERROR: datasets/test.csv not found!")
        print("Please upload the file or check the path.")
        return

    # Load data
    print("\nLoading and preprocessing data...")
    train_dataset, test_dataset, src_vocab, trg_vocab = load_and_preprocess_data(
        'datasets/train.csv',
        'datasets/test.csv'
    )

    print(f"✓ Loaded {len(train_dataset)} training samples")
    print(f"✓ Loaded {len(test_dataset)} test samples")
    print(f"✓ Source vocabulary size: {len(src_vocab)}")
    print(f"✓ Target vocabulary size: {len(trg_vocab)}")

    # Create data loaders
    BATCH_SIZE = 32
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    # Model hyperparameters (must match the trained model)
    INPUT_DIM = len(src_vocab)
    OUTPUT_DIM = len(trg_vocab)
    ENC_EMB_DIM = 256  # Increased for better performance
    DEC_EMB_DIM = 256  # Increased for better performance
    ENC_HID_DIM = 128  # Increased for better performance
    DEC_HID_DIM = 128  # Increased for better performance
    ENC_DROPOUT = 0.5  # Adjusted for better generalization
    DEC_DROPOUT = 0.5  # Adjusted for better generalization
    PAD_IDX = src_vocab.stoi['<pad>']
    SOS_IDX = src_vocab.stoi['<sos>']
    EOS_IDX = src_vocab.stoi['<eos>']

    # Initialize model
    print("\nInitializing model...")
    attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
    enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT, n_layers=2)
    dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn, n_layers=2)
    model = Seq2Seq(enc, dec, PAD_IDX, SOS_IDX, EOS_IDX, device).to(device)

    print(f'Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

    # Load pre-trained weights
    MODEL_NAME = 'models/lstm_baseline.pt'
    if os.path.exists(MODEL_NAME):
        print(f"\nLoading pre-trained weights from {MODEL_NAME}...")
        try:
            model.load_state_dict(torch.load(MODEL_NAME, map_location=device))
            print("✓ Weights loaded successfully!")
        except Exception as e:
            print(f"ERROR loading weights: {e}")
            print("Please make sure the model architecture matches the weights.")
            return
    else:
        print(f"\nERROR: {MODEL_NAME} not found!")
        print("Please upload the pre-trained weights file.")
        return

    # Loss function
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    # Evaluate on test set
    print("\nEvaluating on test set...")
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for i, (src, src_len, trg, trg_len) in enumerate(test_loader):
            src, src_len, trg = src.to(device), src_len.to(device), trg.to(device)
            output, _ = model(src, src_len, trg, 0)

            output = output[1:].view(-1, output.shape[-1])
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)
            test_loss += loss.item()

    test_loss /= len(test_loader)
    test_ppl = math.exp(test_loss)

    print(f'| Test Loss: {test_loss:.3f} | Test PPL: {test_ppl:7.3f} |')

    # Calculate F1-score
    print("\nCalculating F1-score on sample of test data...")
    references = []
    predictions = []

    # Use a subset for faster evaluation (adjust as needed)
    eval_samples = min(100, len(test_dataset))

    for i in range(eval_samples):
        src, trg = test_dataset[i]
        src_text = [src_vocab.itos[idx.item()] for idx in src if idx.item() != PAD_IDX]
        trg_text = [trg_vocab.itos[idx.item()] for idx in trg if idx.item() != PAD_IDX]

        # Remove SOS and EOS tokens
        src_tokens = src_text[1:-1] if len(src_text) > 2 else src_text
        trg_tokens = trg_text[1:-1] if len(trg_text) > 2 else trg_text

        translation, _ = translate_sentence(model, src_tokens, src_vocab, trg_vocab)

        references.append(trg_tokens)
        predictions.append(translation)

    f1_percentage = calculate_f1_score(references, predictions)
    print(f'\nF1-Score: {f1_percentage:.2f}%')

    if f1_percentage < 50:
        print("\n⚠️  WARNING: F1-score is below 50%")
        print("To improve F1-score to 50%+, consider:")
        print("  1. Training with more data")
        print("  2. Using a larger model (increased emb_dim, hid_dim)")
        print("  3. Training for more epochs")
        print("  4. Using pre-trained embeddings (e.g., GloVe)")
        print("  5. Implementing beam search for decoding")
    else:
        print("\n✓ F1-score target (50%) achieved!")

    # Show example translations
    print("\n" + "="*60)
    print("EXAMPLE TRANSLATIONS:")
    print("="*60)

    for i in range(5):
        src, trg = test_dataset[i]
        src_text = [src_vocab.itos[idx.item()] for idx in src if idx.item() != PAD_IDX]
        trg_text = [trg_vocab.itos[idx.item()] for idx in trg if idx.item() != PAD_IDX]

        src_tokens = src_text[1:-1] if len(src_text) > 2 else src_text
        trg_tokens = trg_text[1:-1] if len(trg_text) > 2 else trg_text

        translation, _ = translate_sentence(model, src_tokens, src_vocab, trg_vocab)

        print(f"\nSource (Abstract): {' '.join(src_tokens[:50])}{'...' if len(src_tokens) > 50 else ''}")
        print(f"Target (Title):     {' '.join(trg_tokens)}")
        print(f"Predicted (Title):  {' '.join(translation)}")
        print("-" * 60)

    # Generate submission file
    print("\nGenerating submission file...")
    try:
        submission_data = pd.read_csv('datasets/test.csv')
        abstracts = submission_data['abstract'].values

        titles = []
        for abstract in abstracts:
            tokenized = tokenize(abstract, load_spacy_model())
            title, _ = translate_sentence(model, tokenized, src_vocab, trg_vocab)
            titles.append(' '.join(title).replace('<unk>', ''))

        submission_df = pd.DataFrame({'abstract': abstracts, 'title': titles})
        submission_df.to_csv('datasets/predicted_titles.csv', index=False)
        print(f"✓ Submission file saved to datasets/predicted_titles.csv")
        print(f"  Total predictions: {len(titles)}")

        # Download the submission file
        if 'google.colab' in sys.modules:
            from google.colab import files
            files.download('datasets/predicted_titles.csv')
            print("✓ File downloaded to your local machine")

    except FileNotFoundError:
        print("ERROR: Could not generate submission file")

if __name__ == '__main__':
    main()

Using device: cpu
LSTM BASELINE FOR TITLE GENERATION - INFERENCE MODE

PLEASE UPLOAD THE FOLLOWING FILES:
1. train.csv (with 'abstract' and 'title' columns)
2. test.csv (with 'abstract' column)
3. lstm_baseline.pt (pre-trained model weights)

Upload train.csv...


KeyboardInterrupt: 